In [ ]:
# This script implements the center pull correction.

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import numpy as np
import os
import pickle as pkl
import random
from model import SequenceGenerator

# DATA LOADING
DATASETS_PATH = os.path.join('..', '..', '..','data')
TEST_DATASET_PATH = os.path.join(DATASETS_PATH, 'test.pickle')
TRAIN_DATASET_PATH = os.path.join(DATASETS_PATH, 'train.pickle')

with open(TEST_DATASET_PATH, 'rb') as f:
    test_dataset = pkl.load(f)
    print(test_dataset['label'].value_counts())

with open(TRAIN_DATASET_PATH, 'rb') as f:
    train_dataset = pkl.load(f)
    print(train_dataset['label'].value_counts())

# DATASET CLASS
class SensorDataSet(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __getitem__(self, idx):
        data = self.dataset.iloc[idx]
        x = torch.tensor(data['sensor_data'], dtype=torch.float32)
        y = torch.tensor(data['label'], dtype=torch.long)
        return x, y

    def __len__(self):
        return len(self.dataset)


# TRAINING FUNCTION 
def train_generator(generator, train_loader, num_epochs=10, lr=0.001):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    generator.to(device)
    optimizer = torch.optim.Adam(generator.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    generator.train()
    for epoch in range(num_epochs):
        total_loss = 0
        for x, _ in train_loader:
            x = x.to(device)  # (batch, 128, 6)

            # Input and target for a whole sequence
            input_seq = x[:, :-1, :]  # (batch, 127, 6)
            target_seq = x[:, 1:, :]  # (batch, 127, 6)

            output, _ = generator(input_seq) # Output shape: (batch, 127, 6)

            # Loss between output and target
            loss = loss_fn(output, target_seq)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item() * x.size(0)

        print(f"[Epoch {epoch+1}] Loss: {total_loss / len(train_loader.dataset):.6f}")

# INITIALIZING AND TRAINING
input_dim = 6
hidden_dim = 128
num_layers = 2

generator = SequenceGenerator(input_dim, hidden_dim, num_layers)

train_loader = DataLoader(SensorDataSet(train_dataset), batch_size=32, shuffle=True)

train_generator(generator, train_loader, num_epochs=10, lr=0.001)

In [ ]:
# SEQUENCE GENERATION FUNCTION 
def generate_sequence(generator, seed, mean_seq, std_seq, seq_len=128):
    generator.eval()
    device = next(generator.parameters()).device

    # Format first input as (1, 1, 6) tensor, because the LSTM wants those dimensions
    current_input = torch.tensor(seed, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device) # first seed (1, 1, 6)
    
    generated = [current_input.squeeze(0).squeeze(0)] # Store first sequence step
    hidden = None

    # Regressively generate a sequence by correcting last output to the mean and using it as the next input
    for t in range(seq_len - 1):
        output, hidden = generator(current_input, hidden)  # output: (1, 1, 6)
        
        # Take mean and std of real data at current timestep
        mean_t = torch.tensor(mean_seq[t], dtype=torch.float32).to(device)
        std_t = torch.tensor(std_seq[t], dtype=torch.float32).to(device)

        # Decide a weight for how strong the pull towards mean is
        weight = 1 / (std_t + 1e-6)
        weight = 0.03 * weight / (weight + 1)

        # Pull towards mean
        adjusted_output = output.squeeze(0).squeeze(0) + weight * (mean_t - output.squeeze(0).squeeze(0))

        # Use the adjusted value as input for next step
        current_input = adjusted_output.unsqueeze(0).unsqueeze(0)
        generated.append(adjusted_output)

    return torch.stack(generated).cpu().detach()  # (seq_len, 6)

# PLOTTING FUNCTION
def plot_sequence(dataset, return_figure=True):
    fig, axes = plt.subplots(3,2)

    directions = ["x", "y", "z"]

    value_type = ['acceleration', 'rotation']

    for i, direction_label in enumerate(directions):
        for j, value_label in enumerate(value_type):
            axes[i,j].plot(dataset[...,:,3*j+i])
            if i == 0:
                axes[i,j].set_title(value_label)
                
            if j == 1:
                axes[i,j].text(1, 0.5,direction_label, size=12, rotation=270, transform=axes[i,j].transAxes)
    if return_figure:
        return fig

# PLOT 3 SYNTHETIC SEQUENCES
# calculate mean and std of the real data for correction in generator
train_sensor_data = np.stack(train_dataset['sensor_data'].values)  # (N, 128, 6)
mean_seq = train_sensor_data.mean(axis=0)  # (128, 6)
std_seq = train_sensor_data.std(axis=0)    # (128, 6)

# Collect mean and std of real starting values for generating a new random starting value
start_values = np.stack(train_dataset['sensor_data'].apply(lambda x: x[0]))  # (N, 6)
mean_start = start_values.mean(axis=0)  # (6,)
std_start = start_values.std(axis=0)    # (6,)
std_factor = 0.5

# Generate sequence
for i in range(3):
    # Random starting seed (starting value)
    seed = mean_start + np.random.normal(0, std_start * std_factor, size=mean_start.shape)

    fake_seq = generate_sequence(generator, seed, mean_seq, std_seq)
    plot_sequence(fake_seq.numpy())